# 04 — R2bis Style Premium longitudinal

**Question** : les coefficients de style (`headers`, `lists`, `bold`) augmentent-ils dans le temps ?

**Méthode** : Bradley-Terry style-controlled par mois, avec bootstrap sur chaque cohorte.

**Input** : `data/interim/battles_with_dates.parquet`

**Outputs** :
- `data/processed/style_premium_temporal.parquet`
- `data/processed/style_premium_trends.parquet`
- `paper/figures/R2bis_style_premium_longitudinal.png`

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

from compariawatch.style_premium import (
    CORE_STYLE_FEATURES,
    STYLE_FEATURES,
    compute_monthly_style_premium,
)

INPUT = ROOT / 'data' / 'interim' / 'battles_with_dates.parquet'
OUTPUT = ROOT / 'data' / 'processed' / 'style_premium_temporal.parquet'
TRENDS_OUTPUT = ROOT / 'data' / 'processed' / 'style_premium_trends.parquet'
FIGURE = ROOT / 'paper' / 'figures' / 'R2bis_style_premium_longitudinal.png'
N_BOOT = 40

print('Imports OK')

In [ ]:
df = pd.read_parquet(INPUT)
df = df[df['timestamp'].notna() & df['winner'].isin(['model_a', 'model_b'])].copy()
print(f'Battles R2bis : {len(df):,}')
print(f'Mois : {df["month"].nunique()}')

In [ ]:
style_df = compute_monthly_style_premium(df, n_boot=N_BOOT, style_features=STYLE_FEATURES)
style_df.to_parquet(OUTPUT, index=False)
style_df.head()

In [ ]:
rows = []
for feature, sub in style_df.groupby('feature'):
    sub = sub.sort_values('month_idx')
    x = sm.add_constant(sub['month_idx'])
    y = sub['odds_pct']
    model = sm.OLS(y, x).fit()
    rows.append({
        'feature': feature,
        'beta_odds_pct_per_month': model.params['month_idx'],
        'p_value': model.pvalues['month_idx'],
        'r2': model.rsquared,
        'n_months': len(sub),
        'mean_odds_pct': sub['odds_pct'].mean(),
    })

trends = pd.DataFrame(rows).sort_values('feature').reset_index(drop=True)
trends.to_parquet(TRENDS_OUTPUT, index=False)
trends

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=False)

for ax, feature in zip(axes, CORE_STYLE_FEATURES):
    sub = style_df[style_df['feature'] == feature].sort_values('month_idx')
    trend = trends[trends['feature'] == feature].iloc[0]
    ax.plot(sub['month'], sub['odds_pct'], 'o-', color='steelblue', lw=2)
    ax.fill_between(sub['month'], sub['odds_low'], sub['odds_high'], color='steelblue', alpha=0.18)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f"{feature}\nβ={trend['beta_odds_pct_per_month']:+.2f} pts/mois, p={trend['p_value']:.3f}")
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Effet sur odds de victoire (% / SD)')

fig.suptitle('R2bis — Évolution mensuelle du Style Premium', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE, dpi=200, bbox_inches='tight')
plt.show()
print(FIGURE)